In [1]:
# import necessary libraries
import pandas as pd
import numpy as np

In [2]:
# load the cleaned training dataset
df = pd.read_csv("../data/cleaned_train.csv")

In [9]:
# check data head
df.head().T

,0,1,2,3,4
Age,58.000000,52.000000,56.000000,44.000000,58.000000
Sex,1.000000,1.000000,0.000000,0.000000,1.000000
Chest pain type,4.000000,1.000000,2.000000,3.000000,4.000000
BP,152.000000,125.000000,160.000000,134.000000,140.000000
Cholesterol,239.000000,325.000000,188.000000,229.000000,234.000000
FBS over 120,0.000000,0.000000,0.000000,0.000000,0.000000
EKG results,0.000000,2.000000,2.000000,2.000000,2.000000
Max HR,158.000000,171.000000,151.000000,150.000000,125.000000
Exercise angina,1.000000,0.000000,0.000000,0.000000,1.000000
ST depression,3.600000,0.000000,0.000000,1.000000,3.800000


In [10]:
# check data info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 34 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Age                       630000 non-null  int64  
 1   Sex                       630000 non-null  int64  
 2   Chest pain type           630000 non-null  int64  
 3   BP                        630000 non-null  int64  
 4   Cholesterol               630000 non-null  int64  
 5   FBS over 120              630000 non-null  int64  
 6   EKG results               630000 non-null  int64  
 7   Max HR                    630000 non-null  int64  
 8   Exercise angina           630000 non-null  int64  
 9   ST depression             630000 non-null  float64
 10  Slope of ST               630000 non-null  int64  
 11  Number of vessels fluro   630000 non-null  int64  
 12  Thallium                  630000 non-null  int64  
 13  Heart Disease             630000 non-null  i

In [3]:
"""
Implement level-1 (high impact) features for core predictive signals
"""

# ---- Coronary anatomy ----
df["Any_vessel_disease"] = (df["Number of vessels fluro"] > 0).astype(int)
df["Severe_vessel_disease"] = (df["Number of vessels fluro"] >= 2).astype(int)
df["Abnormal_thallium"] = (df["Thallium"] != 3).astype(int)

# ---- Exercise-induced ischemia ----
df["Exercise_ischemia_flag"] = (
    (df["Exercise angina"] == 1) &
    (df["ST depression"] > 1)
).astype(int)

# ---- Heart rate dynamics ----
df["Heart_rate_reserve"] = df["Max HR"] - (220 - df["Age"])
df["ST_depression_normalized"] = df["ST depression"] / df["Max HR"]

# ---- ST segment severity ----
df["ST_severity_level"] = pd.cut(
    df["ST depression"],
    bins=[-1, 0, 1, 2, np.inf],
    labels=[0, 1, 2, 3]
).astype(int)

# ---- Aggregated risk scores ----
df["Global_heart_risk_index"] = (
    (df["BP"] >= 140).astype(int) +
    (df["Cholesterol"] >= 240).astype(int) +
    (df["FBS over 120"] == 1).astype(int) +
    (df["Exercise angina"] == 1).astype(int) +
    (df["ST depression"] > 1).astype(int) +
    (df["EKG results"] != 0).astype(int) +
    (df["Number of vessels fluro"] > 0).astype(int)
)

df["Metabolic_risk_score"] = (
    (df["BP"] >= 140).astype(int) +
    (df["Cholesterol"] >= 240).astype(int) +
    (df["FBS over 120"] == 1).astype(int)
)

# ---- Non-linear age risk ----
df["Age_squared"] = df["Age"] ** 2


In [4]:
"""
Implement level-2 (medium impact) features for context and refinement
"""

# ---- ECG & chest pain context ----
df["Abnormal_EKG_flag"] = (df["EKG results"] != 0).astype(int)
df["Typical_angina_flag"] = df["Chest pain type"].isin([3, 4]).astype(int)

df["Chest_pain_risk_score"] = df["Chest pain type"].map({
    1: 0,  # asymptomatic
    2: 1,  # atypical angina
    3: 2,  # non-anginal pain
    4: 3   # typical angina
})

# ---- Exercise response nuance ----
df["Low_HR_response_flag"] = (
    df["Max HR"] < 0.85 * (220 - df["Age"])
).astype(int)

df["Exercise_risk_score"] = (
    df["Exercise angina"] +
    (df["ST depression"] > 0).astype(int)
)

# ---- Physiological interactions ----
df["Age_Sex_interaction"] = df["Age"] * df["Sex"]
df["BP_Chol_ratio"] = df["BP"] / df["Cholesterol"]

# ---- Binary threshold indicators ----
df["High_BP_flag"] = (df["BP"] >= 140).astype(int)
df["High_Chol_flag"] = (df["Cholesterol"] >= 240).astype(int)

In [5]:
"""
Implement level-3 (low impact) features for interpretability and redundancy
"""

# ---- Coarse age binning (interpretability-focused) ----
df["Age_group"] = pd.cut(
    df["Age"],
    bins=[0, 40, 50, 60, 100],
    labels=[0, 1, 2, 3]
).astype(int)

In [11]:
# save the engineered dataset as fe_train.csv
df.to_csv("../data/fe_train.csv", index=False)